# Homework 4 - Reinforcement Learning

In [ ]:
import numpy as np
from typing import List, Optional, Tuple, Any, Iterable
import util_rl as util

## Problem 3a: Value Iteration on Number Line MDP

In [ ]:
def value_iteration(
        transitions: np.ndarray,
        rewards: np.ndarray,
        discount: float,
        epsilon: float = 0.001,
        valid_actions: Optional[np.ndarray] = None,
        state_ids: Optional[Iterable[Any]] = None,
        action_ids: Optional[Iterable[Any]] = None,
    ):
    """
    Given transition probabilities and rewards, computes and returns the optimal policy.
    - transitions: np.ndarray with shape (num_states, num_actions, num_states)
    - rewards: np.ndarray with the same shape as transitions
    - epsilon: repeatedly update v until the v values for all states do not change by more than epsilon.
    - valid_actions: optional boolean mask of shape (num_states, num_actions) indicating available actions
    - state_ids: optional iterable mapping each index to a state identifier
    - action_ids: optional iterable mapping each action index to an action identifier
    - Returns: np.ndarray of shape (num_states,) storing the optimal action identifier per state
               (or None if no action is available).
    """
    transitions = np.asarray(transitions, dtype=np.float64)
    rewards = np.asarray(rewards, dtype=np.float64)
    num_states, num_actions, next_state_dim = transitions.shape

    if valid_actions is None:
        action_mask = np.ones((num_states, num_actions), dtype=bool)
    else:
        action_mask = np.asarray(valid_actions, dtype=bool)

    if state_ids is None:
        state_ids = np.arange(num_states)
    else:
        state_ids = np.array(list(state_ids), dtype=object)

    if action_ids is None:
        action_ids = np.arange(num_actions)
    else:
        action_ids = np.array(list(action_ids), dtype=object)

    # Small tie-breaker to prefer larger action indices when Q-values are equal
    tie_breaker = (np.arange(num_actions, dtype=np.float64) * 1e-12)[np.newaxis, :]

    def compute_q(v: np.ndarray) -> np.ndarray:
        """
        Q(s,a) = sum_s' T(s,a,s') * [R(s,a,s') + discount * V(s')]
        Returns: np.ndarray of shape (num_states, num_actions)
        """
        q = np.sum(transitions * (rewards + discount * v[np.newaxis, np.newaxis, :]), axis=2)
        return q

    def compute_policy(q: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Extracts optimal actions and values from Q-table.
        Ties broken by selecting the action with the largest index.
        Returns: (best_actions, best_values) each of shape (num_states,)
        """
        masked_q = np.where(action_mask, q + tie_breaker, -np.inf)
        best_actions = np.argmax(masked_q, axis=1)
        best_values = q[np.arange(num_states), best_actions]
        best_values[~action_mask.any(axis=1)] = 0.0
        return best_actions, best_values

    print('Running value iteration...')

    terminal_mask = ~action_mask.any(axis=1)
    v = np.zeros(num_states, dtype=np.float64)

    while True:
        q = compute_q(v)
        best_actions, new_v = compute_policy(q)
        new_v[terminal_mask] = 0.0

        if np.max(np.abs(new_v - v)) < epsilon:
            v = new_v
            break
        v = new_v

    # Build policy: action identifiers for non-terminal states, None for terminal
    policy = np.full(num_states, None, dtype=object)
    for i in range(num_states):
        if action_mask[i].any():
            policy[i] = action_ids[best_actions[i]]

    return policy

In [ ]:
def run_vi_over_number_line(mdp: util.NumberLineMDP):
    num_states = mdp.num_states
    actions = np.array(list(mdp.actions), dtype=object)
    transitions = np.zeros((num_states, len(actions), num_states), dtype=np.float64)
    rewards = np.zeros_like(transitions)
    valid_actions = np.zeros((num_states, len(actions)), dtype=bool)
    for state_idx, state in enumerate(mdp.indexer.all_states()):
        if state in mdp.terminal_states:
            continue
        valid_actions[state_idx, :] = True
        for action_idx, action in enumerate(actions):
            forward_prob = 0.2 if action == 1 else 0.3
            backward_prob = 1.0 - forward_prob
            forward_state = state + 1
            backward_state = state - 1
            forward_reward = mdp.right_reward if forward_state == mdp.n else (
                mdp.left_reward if forward_state == -mdp.n else mdp.penalty)
            backward_reward = mdp.right_reward if backward_state == mdp.n else (
                mdp.left_reward if backward_state == -mdp.n else mdp.penalty)
            forward_idx = mdp.state_to_index(forward_state)
            backward_idx = mdp.state_to_index(backward_state)
            transitions[state_idx, action_idx, forward_idx] += forward_prob
            rewards[state_idx, action_idx, forward_idx] = forward_reward
            transitions[state_idx, action_idx, backward_idx] += backward_prob
            rewards[state_idx, action_idx, backward_idx] = backward_reward

    state_ids = np.array([mdp.index_to_state(i) for i in range(num_states)], dtype=object)
    pi = value_iteration(
        transitions,
        rewards,
        mdp.discount,
        valid_actions=valid_actions,
        state_ids=state_ids,
        action_ids=actions,
    )
    return state_ids, pi

In [ ]:
# Test value iteration on the NumberLineMDP
mdp = util.NumberLineMDP()
state_ids, pi = run_vi_over_number_line(mdp)

print("\nOptimal policy:")
for state, action in zip(state_ids, pi):
    print(f"  State {state:+d}: action {action}")